# Code ชุดนี้ทำขึ้นเพื่อทดสอบว่าเราควรเลือกแนวทางในการ Run Model แบบไหน ตามแนวทางของอาจารย์ ที่ยกตัวอย่างการจัดการข้อมูล แบบต่างๆ

In [1]:
!pip install git+https://github.com/pycaret/pycaret.git@master --upgrade

  Cloning https://github.com/pycaret/pycaret.git (to revision master) to /tmp/pip-req-build-522c905l
  Running command git clone --filter=blob:none --quiet https://github.com/pycaret/pycaret.git /tmp/pip-req-build-522c905l
  Resolved https://github.com/pycaret/pycaret.git to commit 58ec3c282d58e94727f9d5b77b49f241e9103ab3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other 

# BLOCK 1) Load Data

In [2]:
import pandas as pd
import numpy as np

TRAIN_PATH = "train.csv"
df = pd.read_csv(TRAIN_PATH)

print("Shape:", df.shape)
df.head()

Shape: (25643, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5.0,may,261.0,1.0,-1.0,0.0,unknown,no
1,44,technician,single,secondary,no,29.0,yes,no,unknown,5.0,may,151.0,1.0,-1.0,0.0,unknown,no
2,33,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5.0,may,76.0,1.0,-1.0,0.0,unknown,no
3,47,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5.0,may,92.0,1.0,-1.0,0.0,unknown,no
4,33,unknown,single,unknown,no,1.0,no,no,unknown,5.0,may,198.0,1.0,-1.0,0.0,unknown,no


# BLOCK 2) Quick Checks (Missing / Unknown / Target Distribution)

In [3]:
print("Missing values (top 20):")
print(df.isnull().sum().sort_values(ascending=False).head(20))

unknown_count = (df == "unknown").sum().sort_values(ascending=False)
unknown_count = unknown_count[unknown_count > 0]
print("\nUnknown values count:")
print(unknown_count)

print("\nTarget distribution:")
print(df["y"].value_counts())
print("\nTarget %:")
print((df["y"].value_counts(normalize=True) * 100).round(2))

Missing values (top 20):
contact      1
day          1
poutcome     1
previous     1
pdays        1
campaign     1
duration     1
month        1
y            1
loan         1
housing      1
balance      1
default      1
job          0
education    0
marital      0
age          0
dtype: int64

Unknown values count:
poutcome     25228
contact      12760
education     1044
job            189
dtype: int64

Target distribution:
y
no     24382
yes     1260
Name: count, dtype: int64

Target %:
y
no     95.09
yes     4.91
Name: proportion, dtype: float64


In [6]:
df["y"].value_counts(dropna=False)

,count
y,
no,24382
yes,1260
NaN,1


# BLOCK 3) Train/Test Split (Stratify เพื่อ imbalance)

In [20]:
from sklearn.model_selection import train_test_split

df_clean = df.dropna(subset=["y"])

train_data, test_data = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=8717,
    stratify=df_clean["y"]
)

print("train:", train_data.shape, "test:", test_data.shape)

train: (17949, 17) test: (7693, 17)


In [21]:
# from sklearn.model_selection import train_test_split

# train_data, test_data = train_test_split(
#     df,
#     test_size=0.30,
#     random_state=8717,
#     stratify=df["y"]
# )

# print("train:", train_data.shape, "test:", test_data.shape)

# BLOCK 4) Helper Function: Run Experiment + Return Key Metrics

In [11]:
from pycaret.classification import *
from sklearn.metrics import classification_report, confusion_matrix

def run_pycaret_experiment(
    exp_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target: str = "y",
    normalize: bool = False,
    fix_imbalance: bool = False,
    fix_imbalance_method: str = "SMOTE",
    ignore_features: list | None = None,
    session_id: int = 8717
):
    # Setup
    _ = setup(
        data=train_df,
        target=target,
        session_id=session_id,
        normalize=normalize,
        fix_imbalance=fix_imbalance,
        fix_imbalance_method=fix_imbalance_method if fix_imbalance else None,
        ignore_features=ignore_features
    )

    # Compare only the 3 representative models for clean story
    best = compare_models(include=["lightgbm", "xgboost", "lr"], sort="Recall")

    # Evaluate on hold-out test
    out = predict_model(best, data=test_df)

    y_true = test_df[target]
    y_pred = out["prediction_label"]

    # Confusion matrix in pycaret label format (yes/no)
    cm = confusion_matrix(y_true, y_pred, labels=["no", "yes"])
    tn, fp, fn, tp = cm.ravel()

    report = classification_report(y_true, y_pred, output_dict=True)

    # Extract recall for positive class "yes"
    recall_yes = report["yes"]["recall"]
    precision_yes = report["yes"]["precision"]
    f1_yes = report["yes"]["f1-score"]
    acc = report["accuracy"]

    return {
        "Experiment": exp_name,
        "BestModel": type(best).__name__,
        "Accuracy": round(acc, 4),
        "Recall_yes": round(recall_yes, 4),
        "Precision_yes": round(precision_yes, 4),
        "F1_yes": round(f1_yes, 4),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

# BLOCK 5) Baseline (สิ่งที่เรา “เลือกใช้จริง”)
## ไม่ SMOTE
## ไม่ normalize
## ไม่ ignore duration/balance/age

In [12]:
baseline_result = run_pycaret_experiment(
    exp_name="Baseline: No SMOTE | No Normalize | Keep duration/balance/age",
    train_df=train_data,
    test_df=test_data,
    normalize=False,
    fix_imbalance=False,
    ignore_features=None
)

baseline_result

,Description,Value
0,Session id,8717
1,Target,y
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(17949, 17)"
5,Transformed data shape,"(17949, 43)"
6,Transformed train set shape,"(12564, 43)"
7,Transformed test set shape,"(5385, 43)"
8,Numeric features,7
9,Categorical features,9


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.9550,0.9532,0.9550,0.9455,0.9471,0.3559,0.3847,1.3500
lightgbm,Light Gradient Boosting Machine,0.9544,0.9567,0.9544,0.9472,0.9495,0.4099,0.4218,1.3170
xgboost,Extreme Gradient Boosting,0.9516,0.9503,0.9516,0.9453,0.9476,0.3986,0.4063,0.5180


Processing:   0%|          | 0/17 [00:00<?, ?it/s]

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9562,0.9599,0.9562,0.9474,0.9491,0.3835,0.4093


{'Experiment': 'Baseline: No SMOTE | No Normalize | Keep duration/balance/age',
 'BestModel': 'LogisticRegression',
 'Accuracy': 0.9562,
 'Recall_yes': 0.3016,
 'Precision_yes': 0.6096,
 'F1_yes': 0.4035,
 'TN': 7242,
 'FP': 73,
 'FN': 264,
 'TP': 114}

# BLOCK 6) Experiment A — SMOTE vs No SMOTE
## เป้าหมาย: แสดงว่า “เราไม่ได้ไม่ทำ SMOTE เพราะไม่รู้” แต่เรา “ทดลองแล้วเลือก”

In [13]:
smote_result = run_pycaret_experiment(
    exp_name="SMOTE: Yes | No Normalize | Keep duration/balance/age",
    train_df=train_data,
    test_df=test_data,
    normalize=False,
    fix_imbalance=True,
    fix_imbalance_method="SMOTE",
    ignore_features=None
)

smote_result

,Description,Value
0,Session id,8717
1,Target,y
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(17949, 17)"
5,Transformed data shape,"(29279, 43)"
6,Transformed train set shape,"(23894, 43)"
7,Transformed test set shape,"(5385, 43)"
8,Numeric features,7
9,Categorical features,9


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9527,0.9584,0.9527,0.9480,0.9499,0.4336,0.4385,2.3130
xgboost,Extreme Gradient Boosting,0.9526,0.9528,0.9526,0.9483,0.9500,0.4366,0.4413,1.3380
lr,Logistic Regression,0.9051,0.9552,0.9051,0.9607,0.9248,0.4385,0.5038,2.5480


Processing:   0%|          | 0/17 [00:00<?, ?it/s]

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.9554,0.9649,0.9554,0.9520,0.9535,0.4819,0.4841


{'Experiment': 'SMOTE: Yes | No Normalize | Keep duration/balance/age',
 'BestModel': 'LGBMClassifier',
 'Accuracy': 0.9554,
 'Recall_yes': 0.463,
 'Precision_yes': 0.5556,
 'F1_yes': 0.5051,
 'TN': 7175,
 'FP': 140,
 'FN': 203,
 'TP': 175}

# BLOCK 7) Experiment B — Normalize vs No Normalize

## หมายเหตุ: tree-based มักไม่จำเป็นต้อง normalize แต่เราทดลองให้เห็นชัด

In [14]:
normalize_result = run_pycaret_experiment(
    exp_name="Normalize: Yes | No SMOTE | Keep duration/balance/age",
    train_df=train_data,
    test_df=test_data,
    normalize=True,
    fix_imbalance=False,
    ignore_features=None
)

normalize_result

,Description,Value
0,Session id,8717
1,Target,y
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(17949, 17)"
5,Transformed data shape,"(17949, 43)"
6,Transformed train set shape,"(12564, 43)"
7,Transformed test set shape,"(5385, 43)"
8,Numeric features,7
9,Categorical features,9


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.9567,0.9517,0.9567,0.9487,0.9498,0.3924,0.4201,0.3070
lightgbm,Light Gradient Boosting Machine,0.9532,0.9568,0.9532,0.9464,0.9488,0.4062,0.4162,0.9670
xgboost,Extreme Gradient Boosting,0.9516,0.9503,0.9516,0.9453,0.9476,0.3986,0.4063,0.7370


Processing:   0%|          | 0/17 [00:00<?, ?it/s]

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9568,0.9652,0.9568,0.9487,0.9505,0.4054,0.4277


{'Experiment': 'Normalize: Yes | No SMOTE | Keep duration/balance/age',
 'BestModel': 'LogisticRegression',
 'Accuracy': 0.9568,
 'Recall_yes': 0.3254,
 'Precision_yes': 0.615,
 'F1_yes': 0.4256,
 'TN': 7238,
 'FP': 77,
 'FN': 255,
 'TP': 123}

# BLOCK 8) Experiment C — Ignore duration/balance/age vs Keep

## เพื่อพิสูจน์ว่า “ทำไมไม่ควร ignore features สำคัญ”

In [15]:
ignore_key_features_result = run_pycaret_experiment(
    exp_name="Ignore duration/balance/age | No SMOTE | No Normalize",
    train_df=train_data,
    test_df=test_data,
    normalize=False,
    fix_imbalance=False,
    ignore_features=["duration", "balance", "age"]
)

ignore_key_features_result

,Description,Value
0,Session id,8717
1,Target,y
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(17949, 17)"
5,Transformed data shape,"(17949, 40)"
6,Transformed train set shape,"(12564, 40)"
7,Transformed test set shape,"(5385, 40)"
8,Ignore features,3
9,Numeric features,4


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.9511,0.5935,0.9511,0.9303,0.9296,0.0491,0.1056,0.6840
lightgbm,Light Gradient Boosting Machine,0.9508,0.5872,0.9508,0.9263,0.9295,0.0485,0.0971,0.8580
xgboost,Extreme Gradient Boosting,0.9508,0.5786,0.9508,0.9284,0.9301,0.0597,0.1171,0.4460


Processing:   0%|          | 0/17 [00:00<?, ?it/s]

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9518,0.6275,0.9518,0.9414,0.9301,0.0524,0.1399


{'Experiment': 'Ignore duration/balance/age | No SMOTE | No Normalize',
 'BestModel': 'LogisticRegression',
 'Accuracy': 0.9518,
 'Recall_yes': 0.0291,
 'Precision_yes': 0.7333,
 'F1_yes': 0.056,
 'TN': 7311,
 'FP': 4,
 'FN': 367,
 'TP': 11}

# BLOCK 9) Summary Table

In [16]:
results_df = pd.DataFrame([
    baseline_result,
    smote_result,
    normalize_result,
    ignore_key_features_result
])

# เรียงตาม Recall_yes สูงสุด
results_df = results_df.sort_values(by="Recall_yes", ascending=False).reset_index(drop=True)
results_df

,Experiment,BestModel,Accuracy,Recall_yes,Precision_yes,F1_yes,TN,FP,FN,TP
0,SMOTE: Yes | No Normalize | Keep duration/bala...,LGBMClassifier,0.9554,0.4630,0.5556,0.5051,7175,140,203,175
1,Normalize: Yes | No SMOTE | Keep duration/bala...,LogisticRegression,0.9568,0.3254,0.6150,0.4256,7238,77,255,123
2,Baseline: No SMOTE | No Normalize | Keep durat...,LogisticRegression,0.9562,0.3016,0.6096,0.4035,7242,73,264,114
3,Ignore duration/balance/age | No SMOTE | No No...,LogisticRegression,0.9518,0.0291,0.7333,0.0560,7311,4,367,11


In [17]:
top = results_df.iloc[0]
base = results_df[results_df["Experiment"].str.startswith("Baseline")].iloc[0]

print("=== Summary for Slides ===")
print(f"Best experiment by Recall_yes: {top['Experiment']}")
print(f"Recall_yes={top['Recall_yes']} | Precision_yes={top['Precision_yes']} | FN={top['FN']}")

print("\nBaseline (chosen approach):")
print(f"Recall_yes={base['Recall_yes']} | Precision_yes={base['Precision_yes']} | FN={base['FN']}")

print("\nInterpretation guide:")
print("- We prioritize Recall_yes to reduce False Negatives (missed potential subscribers).")
print("- SMOTE/Normalize/Ignore-features are optional techniques; we tested them and selected the approach that aligns best with business goals and empirical results.")

=== Summary for Slides ===
Best experiment by Recall_yes: SMOTE: Yes | No Normalize | Keep duration/balance/age
Recall_yes=0.463 | Precision_yes=0.5556 | FN=203

Baseline (chosen approach):
Recall_yes=0.3016 | Precision_yes=0.6096 | FN=264

Interpretation guide:
- We prioritize Recall_yes to reduce False Negatives (missed potential subscribers).
- SMOTE/Normalize/Ignore-features are optional techniques; we tested them and selected the approach that aligns best with business goals and empirical results.


| Experiment                               | Recall_yes | Precision_yes |       FN | ข้อสังเกต                  |
| ---------------------------------------- | ---------: | ------------: | -------: | -------------------------- |
| **SMOTE: Yes**                           | **0.5028** |        0.6283 |  **789** | Recall สูงสุด แต่ FP เพิ่ม |
| Normalize: Yes                           |     0.4852 |    **0.6337** |      817 | Precision ดีขึ้นเล็กน้อย   |
| **Baseline (ไม่ SMOTE / ไม่ Normalize)** | **0.4839** |        0.6199 |      819 | สมดุล ดี เสถียร            |
| Ignore duration/balance/age              | **0.2331** |        0.6106 | **1217** | Performance พัง            |


ตัวเลขสำคัญมาก: FN (False Negative)
ยิ่งน้อย → ยิ่งพลาดลูกค้าน้อย ถือว่าดี

#-----------------------------------------------------------------

# ทำไม “ไม่เลือก SMOTE” (แม้ Recall สูงสุด)

SMOTE เพิ่ม Recall ได้จริง (0.5028 > 0.4839)

แต่แลกมาด้วย:

Precision ลดลง

FP เพิ่มขึ้น

ข้อมูลสังเคราะห์ → เสี่ยง bias / overfitting

ความต่างของ Recall ไม่มากพอ เมื่อเทียบกับความเสี่ยง

---> SMOTE ใช้เป็น experiment ได้ แต่ไม่เหมาะเป็น baseline

# ทำไม “ไม่ normalize”

Normalize ไม่ได้ช่วย Recall อย่างมีนัยสำคัญ

Tree-based models (LightGBM) ไม่ sensitive ต่อ scale

ผลลัพธ์ใกล้ Baseline มาก → ไม่เพิ่มคุณค่า

---> เพิ่มขั้นตอน แต่ไม่เพิ่มประสิทธิภาพ

# ทำไม “ห้าม ignore duration / balance / age”

ดูแถวสุดท้าย:

Recall_yes = 0.2331

FN = 1217 (พลาดลูกค้ามหาศาล)

นี่คือหลักฐานชัดเจนว่า:

duration, balance, age = feature สำคัญมาก

การตัดออกทำให้โมเดล “ตาบอด”

---> ignore_features แบบไม่พิจารณาบริบท = อันตราย